# Dual-GPU sharded runner (both Kaggle T4s)

Runs an experiment grid across **both T4 GPUs** by sharding the run-list into two subprocesses, each
pinned to one GPU via `CUDA_VISIBLE_DEVICES` (so `train.py` needs no change — each process sees its card
as `cuda:0`). Each shard writes its own CSV (no write race); we then merge + push to S3. Resumable:
completed runs (pulled from S3) are skipped.

Default grid = the **Tier-A feature ablation** (MambaLOB over all feature sets + a TLOB spot-check), but
the `GRID` cell is generic — drop in any list of run specs (e.g. Tier-B architectures later).

Setup: enable **GPU T4 x2**; secrets `GH_PAT`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`.
Clean S3 layout: `experiments/feature_ablation/{results,checkpoints,figures}/`.


## 1. Runtime check (expect 2 GPUs)

In [ ]:
import torch, platform
print("Python:", platform.python_version(), "| Torch:", torch.__version__)
n = torch.cuda.device_count()
print(f"GPUs visible: {n}")
for i in range(n):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")
assert n >= 1, "Enable GPU. For 2x speedup choose 'GPU T4 x2'."
if n < 2:
    print("Only 1 GPU -> will run a single shard (still works, no speedup).")

## 2. Get the project code (GH_PAT secret)

In [ ]:
import sys, subprocess, pathlib
REPO_URL = "https://github.com/rajjoseph48/nse-lob-capstone.git"
REPO_DIR = "nse-lob-capstone"
def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    import os
    return os.environ.get(name, "")
GITHUB_TOKEN = _get_secret("GH_PAT")
def add_modeling(base):
    base = pathlib.Path(base)
    for cand in (base / "modeling", base):
        if (cand / "models.py").exists():
            sys.path.insert(0, str(cand.resolve()))
            return str(cand.resolve())
    return None
MODELING_DIR = None
for c in (".", REPO_DIR, "/kaggle/working/" + REPO_DIR, "/content/" + REPO_DIR):
    MODELING_DIR = add_modeling(c)
    if MODELING_DIR:
        break
if not MODELING_DIR:
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
    MODELING_DIR = add_modeling(REPO_DIR)
print("modeling/ dir:", MODELING_DIR)

## 3. Install deps (Mamba kernel + boto3)

In [ ]:
import subprocess, sys
def pipq(*pkgs): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pipq("boto3")
pipq("ninja", "packaging", "setuptools", "wheel")
pipq("--no-build-isolation", "causal-conv1d")
pipq("--no-build-isolation", "mamba-ssm")
try:
    import mamba_ssm
    print("install step done | mamba-ssm", mamba_ssm.__version__)
except Exception as e:
    print("install step done | mamba-ssm NOT importable -> pure-PyTorch fallback:", repr(e))

## 4. Download NSE data from S3 (shared by both shards)

In [ ]:
import os, boto3, pathlib
os.environ["AWS_ACCESS_KEY_ID"] = _get_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = _get_secret("AWS_SECRET_ACCESS_KEY")
BUCKET, PREFIX, REGION = "lob-capstone-data", "lob-data/dhan/", "ap-south-2"
DATA_DIR_NSE = "nse_data/dhan"; pathlib.Path(DATA_DIR_NSE).mkdir(parents=True, exist_ok=True)
s3 = boto3.client("s3", region_name=REGION)
n = 0
for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX).get("Contents", []):
    if o["Size"] == 0:
        continue
    dst = os.path.join(DATA_DIR_NSE, o["Key"].split("/")[-1])
    if not os.path.exists(dst):
        s3.download_file(BUCKET, o["Key"], dst)
    n += 1
print(f"{n} parquet files in {DATA_DIR_NSE}")

## 5. Build the run-list + resume
`GRID` is the experiment list — edit freely. We pull the merged results from S3 and keep only the
not-yet-done specs, then write `runlist.json` for the shards.

In [ ]:
import json, pathlib, pandas as pd
from nbenv import s3_client, s3_pull_area

AREA = "feature_ablation"
MERGED = pathlib.Path("results/feature_ablation.csv")

GRID = []
for fs in ["base40", "micro", "orders60", "depth80", "all"]:
    for h in [10, 100]:
        GRID.append({"model": "mambalob", "symbol": "NIFTY", "feature_set": fs,
                     "horizon": h, "label_scheme": "A", "seed": 0})
for fs in ["base40", "micro"]:                       # SOTA spot-check
    GRID.append({"model": "tlob", "symbol": "NIFTY", "feature_set": fs,
                 "horizon": 10, "label_scheme": "A", "seed": 0})

_s3 = s3_client(); s3_pull_area(_s3, MERGED, AREA)
done = set()
if MERGED.exists():
    d = pd.read_csv(MERGED)
    done = {(r.model, r.symbol, r.feature_set, int(r.horizon), r.label_scheme, int(r.seed))
            for r in d.itertuples()}
def _key(s):
    return (s["model"], s["symbol"], s["feature_set"], int(s["horizon"]), s["label_scheme"], int(s["seed"]))
todo = [s for s in GRID if _key(s) not in done]
pathlib.Path("runlist.json").write_text(json.dumps(todo))
print(f"grid={len(GRID)} | done={len(done)} | to run this session={len(todo)}")
todo

## 6. Launch one shard per GPU (concurrent) and stream logs
Each shard is a subprocess pinned to one GPU. We tail both logs until they finish.

In [ ]:
import os, sys, subprocess, time, torch

n_gpus = torch.cuda.device_count()
nshards = max(1, n_gpus)
RUN_SHARD = f"{MODELING_DIR}/run_shard.py"
pathlib.Path("results").mkdir(exist_ok=True)

procs = []
for i in range(nshards):
    env = dict(os.environ); env["CUDA_VISIBLE_DEVICES"] = str(i)
    logf = f"shard{i}.log"; fh = open(logf, "w")
    cmd = [sys.executable, RUN_SHARD, "--runlist", "runlist.json", "--shard", str(i),
           "--nshards", str(nshards), "--data-dir", DATA_DIR_NSE, "--out", f"results/shard{i}.csv",
           "--ckpt-dir", "checkpoints/featabl", "--area", "feature_ablation", "--epochs", "20"]
    procs.append([subprocess.Popen(cmd, env=env, stdout=fh, stderr=subprocess.STDOUT), fh, logf, 0])
    print(f"launched shard {i} on CUDA_VISIBLE_DEVICES={i}")

while any(p[0].poll() is None for p in procs):
    for p in procs:
        with open(p[2]) as r:
            r.seek(p[3]); chunk = r.read(); p[3] = r.tell()
        if chunk:
            print(chunk, end="")
    time.sleep(10)
for p in procs:                                        # final flush
    p[1].close()
    with open(p[2]) as r:
        r.seek(p[3]); print(r.read(), end="")
print("\nexit codes:", [p[0].poll() for p in procs])

## 7. Merge shard CSVs + push to S3

In [ ]:
import glob, pandas as pd
from nbenv import s3_client, s3_put_area
keys = ["model", "symbol", "feature_set", "horizon", "label_scheme", "seed"]
frames = [pd.read_csv(MERGED)] if MERGED.exists() else []
frames += [pd.read_csv(f) for f in glob.glob("results/shard*.csv")]
alldf = pd.concat(frames, ignore_index=True).drop_duplicates(subset=keys, keep="last") if frames else pd.DataFrame()
alldf.to_csv(MERGED, index=False)
s3_put_area(s3_client(), MERGED, AREA)
print(f"merged {len(alldf)} rows -> {MERGED}  (s3: experiments/{AREA}/results/)")
alldf.sort_values(keys)

## 8. Findings — does microstructure help? (MambaLOB, delta vs base40)

In [ ]:
import matplotlib.pyplot as plt
res = pd.read_csv(MERGED)
ORDER = ["base40", "micro", "orders60", "depth80", "all"]
mam = res[res.model == "mambalob"]
piv = mam.pivot_table(index="feature_set", columns="horizon", values="test_weighted_f1").reindex(ORDER)
print("MambaLOB (NIFTY) weighted-F1 by feature set x horizon:")
print(piv.round(4).to_string())
print("\nDelta vs base40 (positive = features help):")
print((piv - piv.loc["base40"]).round(4).to_string())
ax = piv.plot(marker="o"); ax.set(title="MambaLOB NIFTY — weighted-F1 by feature set",
    xlabel="feature set", ylabel="weighted-F1"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("results/fig_feature_ablation.png", dpi=150, bbox_inches="tight")
s3_put_area(s3_client(), "results/fig_feature_ablation.png", AREA); plt.show()
if (res.model == "tlob").any():
    print("\nTLOB spot-check (NIFTY k=10):")
    print(res[res.model == "tlob"][["feature_set", "test_weighted_f1", "test_macro_f1"]].to_string(index=False))